# GeoSPARQL 1.0 Tutorial (rdflib-geosparql)
This tutorial showcases the functions of the GeoSPARQL 1.0 standard which have been implemented in rdflib-geosparql.

## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import shapely.geometry
from shapely import wkt
from shapely.ops import transform
from shapely.ops import unary_union
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, FullScreenControl, LayersControl, Popup#, Tooltip

mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in resultlist:
            for item in row:
                if isinstance(row[item],Literal) and row[item].datatype!=None:
                    res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</a></td></tr>"
                elif isinstance(row[item],URIRef):
                    res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
                else:
                    res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    bboxes=[]
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in rows:
            if resultlist!=None and row in resultlist[0]:
                popup="<h3>"+str(row)+"</h3><ul>"        
                for rowres in resultlist:
                    for item in rowres:
                        if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip()+"</a></li>"
                        elif isinstance(rowres[item],URIRef):
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                        else:
                            popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip()+"</li>"
                popup+="</ul>"
                toprocess=resultlist[0][row].strip()
                if toprocess.startswith("<http"):
                    toprocess=toprocess[toprocess.find(">")+1:]
                geom = wkt.loads(toprocess)
                if not shapely.is_empty(geom):
                    lastcentroid=geom.centroid
                    nlayer=WKTLayer(name=str(row),wkt_string=geom.wkt,hover_style={"fillColor": "red"})
                    bounds=geom.bounds
                    bboxes.append(shapely.geometry.box(bounds[0],bounds[1],bounds[2],bounds[3]))
                    msg=W.HTML()
                    msg.value=popup
                    nlpopup=Popup(
                        location=[lastcentroid.y,lastcentroid.x],
                        child=msg,
                        close_button=True,
                        auto_close=False,
                        close_on_escape_key=False
                    )
                    nlayer.popup=msg
                    layers.append(nlayer)
    if lmap==None:
        maxbounds=unary_union(bboxes).bounds
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
        control = LayersControl(position='topright')
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    lmap.fit_bounds(((maxbounds[1],maxbounds[2]),(maxbounds[3],maxbounds[0])))
    return lmap      

def processQueryResults(qres,rows=[],lmap=None):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-ju5safc6.log


## GeoSPARQL 1.0 Egenhofer Relations

### [geof:ehContains](http://www.opengis.net/def/function/geosparql/ehContains) Function

Checks whether two input geometry meet the conditions of the [geo:ehContains](http://www.opengis.net/ont/geosparql#ehContains) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?fLiteral (xsd:boolean(?ehContains) as ?AcontainsF)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:ehContains(?aLiteral, ?fLiteral) as ?ehContains)
}
"""),["aLiteral","fLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
AcontainsF,true


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:ehCovers](http://www.opengis.net/def/function/geosparql/ehCovers) Function

Checks whether two input geometry meet the conditions of the [geo:ehCovers](http://www.opengis.net/ont/geosparql#ehCovers) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehCovers) as ?AcoversB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:ehCovers(?aLiteral, ?bLiteral) as ?ehCovers)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AcoversB,true


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:ehCoveredBy](http://www.opengis.net/def/function/geosparql/ehCoveredBy) Function

Checks whether two input geometry meet the conditions of the [geo:ehCoveredBy](http://www.opengis.net/ont/geosparql#ehCoveredBy) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehCoveredBy) as ?BcoveredByA)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:ehCoveredBy(?bLiteral, ?aLiteral) as ?ehCoveredBy)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
BcoveredByA,true


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:ehDisjoint](http://www.opengis.net/def/function/geosparql/ehDisjoint) Function

Checks whether two input geometry meet the conditions of the [geo:ehDisjoint](http://www.opengis.net/ont/geosparql#ehDisjoint) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [5]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehDisjoint) as ?BdisjointB) (xsd:boolean(?ehDisjoint2) as ?BdisjointC)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:ehDisjoint(?bLiteral, ?bLiteral) as ?ehDisjoint)
  BIND (geof:ehDisjoint(?bLiteral, ?cLiteral) as ?ehDisjoint2)
}
"""),["bLiteral","cLiteral"],None)

Variable,Value
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
BdisjointB,false
BdisjointC,true


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:ehEquals](http://www.opengis.net/def/function/geosparql/ehEquals) Function

Checks whether two input geometry meet the conditions of the [geo:ehEquals](http://www.opengis.net/ont/geosparql#ehEquals) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?ehEquals) as ?AequalsA) (xsd:boolean(?ehEquals2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:ehEquals(?aLiteral, ?aLiteral) as ?ehEquals)
  BIND (geof:ehEquals(?aLiteral, ?bLiteral) as ?ehEquals2)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:ehInside](http://www.opengis.net/def/function/geosparql/ehInside) Function

Checks whether two input geometry meet the conditions of the [geo:ehInside](http://www.opengis.net/ont/geosparql#ehInside) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [7]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?insidee) as ?FinsideA)
WHERE {
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:ehInside(?fLiteral, ?aLiteral) as ?insidee)
}
"""),["aLiteral","fLiteral"],None)

True


Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
FinsideA,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:ehMeet](http://www.opengis.net/def/function/geosparql/ehMeet) Function

Checks whether two input geometry meet the conditions of the [geo:ehMeet](http://www.opengis.net/ont/geosparql#ehMeet) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cLiteral (xsd:boolean(?ehMeet) as ?AmeetC)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:ehMeet(?aLiteral, ?cLiteral) as ?ehMeet)
}
"""),["aLiteral","cLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
AmeetC,true


Map(center=[34.4, -83.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:ehOverlap](http://www.opengis.net/def/function/geosparql/ehOverlap) Function

Checks whether two input geometry meet the conditions of the [geo:ehOverlap](http://www.opengis.net/ont/geosparql#ehOverlap) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [9]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?ehOverlap) as ?AoverlapD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:ehOverlap(?aLiteral, ?dLiteral) as ?ehOverlap)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
AoverlapD,true


Map(center=[34.099999999999994, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## GeoSPARQL 1.0 RCC8 Relations

### [geof:rcc8dc](http://www.opengis.net/def/function/geosparql/rcc8dc) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8dc](http://www.opengis.net/ont/geosparql#rcc8dc) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [10]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?bLiteral ?cLiteral (xsd:boolean(?rcc8dc) as ?BdisjointB) (xsd:boolean(?rcc8dc2) as ?BdisjointC)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:rcc8dc(?bLiteral, ?bLiteral) as ?rcc8dc)
  BIND (geof:rcc8dc(?bLiteral, ?cLiteral) as ?rcc8dc2)
}
"""),["bLiteral","cLiteral"],None)

Variable,Value
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
BdisjointB,false
BdisjointC,true


Map(center=[34.4, -83.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:rcc8ec](http://www.opengis.net/def/function/geosparql/rcc8ec) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8ec](http://www.opengis.net/ont/geosparql#rcc8ec) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cLiteral (xsd:boolean(?sfTouches) as ?AtouchesC)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:rcc8ec(?aLiteral, ?cLiteral) as ?sfTouches)
}
"""),["aLiteral","cLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
AtouchesC,true


Map(center=[34.4, -83.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:rcc8eq](http://www.opengis.net/def/function/geosparql/rcc8eq) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8eq](http://www.opengis.net/ont/geosparql#rcc8eq) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?rcc8eq) as ?AequalsA) (xsd:boolean(?rcc8eq2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:rcc8eq(?aLiteral, ?aLiteral) as ?rcc8eq)
  BIND (geof:rcc8eq(?aLiteral, ?bLiteral) as ?rcc8eq2)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:rcc8ntpp](http://www.opengis.net/def/function/geosparql/rcc8ntpp) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8ntpp](http://www.opengis.net/ont/geosparql#rcc8ntpp) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [13]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gLiteral (xsd:boolean(?rcc8ntppr) as ?Grcc8ntppA) (xsd:boolean(?rcc8ntppr2) as ?Arcc8ntppG)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:G geo:hasDefaultGeometry ?gGeom .
  ?gGeom geo:asWKT ?gLiteral .
  BIND (geof:rcc8ntpp(?gLiteral, ?aLiteral) as ?rcc8ntppr)
  BIND (geof:rcc8ntpp(?aLiteral, ?gLiteral) as ?rcc8ntppr2)
}
"""),["aLiteral","gLiteral"],None)

True
False


Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))"
Grcc8ntppA,true
Arcc8ntppG,false


Map(center=[34.300000000000004, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

### [geof:rcc8ntppi](http://www.opengis.net/def/function/geosparql/rcc8ntppi) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8ntppi](http://www.opengis.net/ont/geosparql#rcc8ntppi) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gLiteral (xsd:boolean(?rcc8ntppir) as ?Arcc8ntppiG) (xsd:boolean(?rcc8ntppir2) as ?Grcc8ntppiA)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:G geo:hasDefaultGeometry ?gGeom .
  ?gGeom geo:asWKT ?gLiteral .
  BIND (geof:rcc8ntppi(?aLiteral, ?gLiteral) as ?rcc8ntppir)
  BIND (geof:rcc8ntppi(?gLiteral, ?aLiteral) as ?rcc8ntppir2)
}
"""),["aLiteral","gLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.5 34.2, -83.3 34.2, -83.3 34.4, -83.5 34.4, -83.5 34.2))"
Arcc8ntppiG,true
Grcc8ntppiA,false


Map(center=[34.300000000000004, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

### [geof:rcc8po](http://www.opengis.net/def/function/geosparql/rcc8po) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8po](http://www.opengis.net/ont/geosparql#rcc8po) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [15]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?rcc8por) as ?Arcc8poD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:rcc8po(?aLiteral, ?dLiteral) as ?rcc8por)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
Arcc8poD,true


Map(center=[34.099999999999994, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:rcc8tpp](http://www.opengis.net/def/function/geosparql/rcc8tpp) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8tpp](http://www.opengis.net/ont/geosparql#rcc8tpp) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [16]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?rcc8tppr) as ?Arcc8tppB)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?aGeom geo:asWKT ?bLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:rcc8tpp(?aLiteral, ?bLiteral) as ?rcc8tppr)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
Arcc8tppB,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:rcc8tppi](http://www.opengis.net/def/function/geosparql/rcc8tppi) Function

Checks whether two input geometry meet the conditions of the [geo:rcc8tppi](http://www.opengis.net/ont/geosparql#rcc8tppi) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [17]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?rcc8tppir) as ?Arcc8tppiB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:rcc8tppi(?aLiteral, ?bLiteral) as ?rcc8tppir)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
Arcc8tppiB,true


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## GeoSPARQL 1.0 Simple Feature Relations

### [geof:sfContains](http://www.opengis.net/def/function/geosparql/sfContains) Function

Checks whether two input geometry meet the conditions of the [geo:sfContains](http://www.opengis.net/ont/geosparql#sfContains) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [18]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?fLiteral (xsd:boolean(?sfContains) as ?AcontainsF)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:sfContains(?aLiteral, ?fLiteral) as ?sfContains)
}
"""),["aLiteral","fLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
AcontainsF,true


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:sfCrosses](http://www.opengis.net/def/function/geosparql/sfCrosses) Function

Checks whether two input geometry meet the conditions of the [geo:sfCrosses](http://www.opengis.net/ont/geosparql#sfCrosses) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [19]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?eLiteral (xsd:boolean(?sfCrosses) as ?EcrossesA)
WHERE {
  my:E geo:hasDefaultGeometry ?eGeom .
  ?eGeom geo:asWKT ?eLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:sfCrosses(?eLiteral, ?aLiteral) as ?sfCrosses)
}
"""),["aLiteral","eLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
eLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"
EcrossesA,true


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### [geof:sfDisjoint](http://www.opengis.net/def/function/geosparql/sfDisjoint) Function

Checks whether two input geometry meet the conditions of the [geo:sfDisjoint](http://www.opengis.net/ont/geosparql#sfDisjoint) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [20]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral (xsd:boolean(?sfDisjoint) as ?CdisjointF)
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:sfDisjoint(?cLiteral, ?fLiteral) as ?sfDisjoint)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
CdisjointF,true


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:sfEquals](http://www.opengis.net/def/function/geosparql/sfEquals) Function

Checks whether two input geometry meet the conditions of the [geo:sfEquals](http://www.opengis.net/ont/geosparql#sfEquals) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [21]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?sfEquals) as ?AequalsA) (xsd:boolean(?sfEquals2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:sfEquals(?aLiteral, ?aLiteral) as ?sfEquals)
  BIND (geof:sfEquals(?aLiteral, ?bLiteral) as ?sfEquals2)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:sfIntersects](http://www.opengis.net/def/function/geosparql/sfIntersects) Function

Checks whether two input geometry meet the conditions of the [geo:sfIntersects](http://www.opengis.net/ont/geosparql#sfIntersects) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [22]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?sfIntersects) as ?AintersectsD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:sfIntersects(?aLiteral, ?dLiteral) as ?sfIntersects)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
AintersectsD,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:sfOverlaps](http://www.opengis.net/def/function/geosparql/sfOverlaps) Function

Checks whether two input geometry meet the conditions of the [geo:sfOverlaps](http://www.opengis.net/ont/geosparql#sfOverlaps) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [23]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?sfOverlaps) as ?AoverlapsD)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:sfOverlaps(?aLiteral, ?dLiteral) as ?sfOverlaps)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
AoverlapsD,true


Map(center=[34.099999999999994, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:sfTouches](http://www.opengis.net/def/function/geosparql/sfTouches) Function

Checks whether two input geometry meet the conditions of the [geo:sfTouches](http://www.opengis.net/ont/geosparql#sfTouches) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [24]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cLiteral (xsd:boolean(?sfTouches) as ?AtouchesC)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  BIND (geof:sfTouches(?aLiteral, ?cLiteral) as ?sfTouches)
}
"""),["aLiteral","cLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
AtouchesC,true


Map(center=[34.4, -83.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:sfWithin](http://www.opengis.net/def/function/geosparql/sfWithin) Function

Checks whether two input geometry meet the conditions of the [geo:sfWithin](http://www.opengis.net/ont/geosparql#sfWithin) relation and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [25]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?sfWithin) as ?BwithinC)
WHERE {
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:sfWithin(?bLiteral, ?aLiteral) as ?sfWithin)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
BwithinC,true


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## GeoSPARQL 1.0 Non-topological query functions

### [geof:boundary](http://www.opengis.net/def/function/geosparql/boundary) Function

Returns the boundary of a given geometry as a LineString in the literal format and the CRS of the input geometry.

In [26]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?boundary
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundary(?aLiteral) as ?boundary)
}
"""),["aLiteral","boundary"],None)

Variable,Value
boundary,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### [geof:buffer](http://www.opengis.net/def/function/geosparql/buffer) Function

In [27]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?buffer
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:buffer(?aLiteral,"5.0"^^xsd:double,<http://www.ontology-of-units-of-measure.org/resource/om-2/meter>) as ?buffer)
}
"""),["aLiteral","buffer"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
buffer,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-88.6 34.1, -88.6 34.5, -88.50392640201615 35.47545161008064, -88.21939766255643 36.41341716182545, -87.75734806151272 37.277851165098014, -87.13553390593273 38.03553390593274, -86.37785116509801 38.65734806151273, -85.51341716182544 39.11939766255644, -84.57545161008063 39.403926402016154, -83.6 39.5, -83.2 39.5, -82.22454838991936 39.403926402016154, -81.28658283817455 39.11939766255644, -80.42214883490199 38.65734806151273, -79.66446609406727 38.03553390593274, -79.04265193848728 37.277851165098014, -78.58060233744357 36.41341716182545, -78.29607359798385 35.47545161008064, -78.2 34.5, -78.2 34.1, -78.29607359798385 33.12454838991936, -78.58060233744357 32.18658283817455, -79.04265193848728 31.32214883490199, -79.66446609406727 30.564466094067264, -80.42214883490199 29.942651938487273, -81.28658283817455 29.480602337443568, -82.22454838991936 29.196073597983847, -83.2 29.1, -83.6 29.1, -84.57545161008063 29.196073597983847, -85.51341716182544 29.480602337443568, -86.37785116509801 29.942651938487273, -87.13553390593273 30.564466094067264, -87.75734806151272 31.32214883490199, -88.21939766255643 32.18658283817455, -88.50392640201615 33.12454838991936, -88.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999998], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:convexHull](http://www.opengis.net/def/function/geosparql/convexHull) Function

Calculates the convex hull of a given geometry in the CRS and literal format of the input geometry.

In [28]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?chull
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:convexHull(?aLiteral) as ?chull)
}
"""),["aLiteral","chull"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
chull,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### [geof:difference](http://www.opengis.net/def/function/geosparql/difference) Function

Calculates the difference between two geometries as a new geometry.

The result is returned in the literal format of the first input geometry.

In [29]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?difference
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:difference(?aLiteral, ?dLiteral) as ?difference)
}
"""),["aLiteral","dLiteral","difference"],None)

[(<POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))>, 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'), (<POLYGON ((-83.3 34, -83.1 34, -83.1 34.2, -83.3 34.2, -83.3 34))>, 'http://www.opengis.net/def/crs/OGC/1.3/CRS84')]
POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1))


Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
difference,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1))"


Map(center=[34.309999999999995, -83.41], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_t…

### [geof:distance](http://www.opengis.net/def/function/geosparql/distance) Function

Calculates the minimum distance between two GeoSPARQL geometries in a given unit as an [xsd:double](http://www.w3.org/2001/XMLSchema#double).

In [30]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?distance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:distance(?cLiteral, ?fLiteral,"http://www.opengis.net/def/uom/OGC/1.0/meter"^^xsd:anyURI) as ?distance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
distance,0.20000000000000284


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:envelope](http://www.opengis.net/def/function/geosparql/envelope) Function 🧊

Returns the envelope of a given geometry in the literal format and CRS of the input geometry.

In [31]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?envelope ?envelope3d
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:envelope(?aLiteral) as ?envelope)
  BIND (geof:envelope("Polygon Z((-83.6 34.1 5.0, -83.2 34.1 5.0, -83.2 34.5 5.0, -83.6 34.5 5.0, -83.6 34.1 5.0))"^^geo:wktLiteral) as ?envelope3d)
}
"""),["aLiteral","envelope"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
envelope,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
envelope3d,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON Z ((-83.6 34.1 5, -83.2 34.5 5, -83.6 34.5 5, -83.6 34.1 5, -83.2 34.1 5, -83.2 34.5 5, -83.6 34.5 5, -83.6 34.1 5))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:geometryType](http://www.opengis.net/def/function/geosparql/geometryType) Function

Returns the assigned geometry type of an input geometry as a [xsd:string](http://www.w3.org/2001/XMLSchema#string).

In [32]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?gtype
WHERE {
  my:A geo:hasDefaultGeometry ?geom .
  ?geom geo:asWKT ?literal .
  BIND (geof:geometryType(?literal) as ?gtype)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gtype,Polygon


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:getSRID](http://www.opengis.net/def/function/geosparql/getSRID) Function

Returns the SRID of a geometry. The SRID is returned as an [xsd:anyURI](http://www.w3.org/2001/XMLSchema#anyURI)

In [33]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?srid
WHERE {
  my:A geo:hasDefaultGeometry ?geom .
  ?geom geo:asWKT ?literal .
  BIND (geof:getSRID(?literal) as ?srid)
}
"""),["literal","envelope"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
srid,http://www.opengis.net/def/crs/OGC/1.3/CRS84


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:intersection](http://www.opengis.net/def/function/geosparql/intersection) Function

Calculates the intersection of two geometries. Results are returned in the literal format and CRS of the first input geometry

In [34]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?intersection
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:intersection(?aLiteral, ?dLiteral) as ?intersection)
}
"""),["aLiteral","dLiteral","intersection"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
intersection,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.2 34.1, -83.3 34.1, -83.3 34.2, -83.2 34.2, -83.2 34.1))"


Map(center=[34.15, -83.25], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### [geof:relate](http://www.opengis.net/def/function/geosparql/relate) Function

In [35]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?relate) as ?relates) (xsd:boolean(?relate2) as ?relates2)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  # "T*****FF*" refers to a 'contains' relation in DE-9IM
  BIND (geof:relate(?aLiteral, ?bLiteral, "T*****FF*"^^xsd:string) as ?relate)
  BIND (geof:relate(?aLiteral, ?bLiteral, "F*****FF*"^^xsd:string) as ?relate2)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
relates,true
relates2,false


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:symDifference](http://www.opengis.net/def/function/geosparql/symDifference) Function

Calculates the symmetric difference of two geometries. Results are returned in the literal format and CRS of the first input geometry

In [36]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?sdifference
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:symDifference(?aLiteral, ?dLiteral) as ?sdifference)
}
"""),["aLiteral","dLiteral","sdifference"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
sdifference,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> MULTIPOLYGON (((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.3 34.2, -83.3 34.1, -83.6 34.1)), ((-83.2 34.1, -83.2 34.2, -83.1 34.2, -83.1 34, -83.3 34, -83.3 34.1, -83.2 34.1)))"


Map(center=[34.27222222222222, -83.3722222222222], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### [geof:union](http://www.opengis.net/def/function/geosparql/union) Function

Calculates the union of two geometries. Results are returned in the literal format and CRS of the first input geometry

In [37]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?union
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:union(?aLiteral, ?dLiteral) as ?union)
}
"""),["aLiteral","dLiteral","union"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
union,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.2, -83.1 34.2, -83.1 34, -83.3 34, -83.3 34.1, -83.6 34.1))"


Map(center=[34.26578947368421, -83.36578947368422], controls=(ZoomControl(options=['position', 'zoom_in_text',…